# Group-Aware Classification of Mouse Protein Expression

## Objective

This notebook develops a baseline machine-learning model for predicting the eight experimental classes from 77 protein-expression measurements.

Because each biological mouse contributes 15 repeated measurements, all measurements from the same mouse must remain together during model evaluation. This prevents the model from being tested on mice it has already encountered during training.

Missing-value imputation, feature scaling, and model fitting will be performed using training data only.

## 1. Load the modelling data

The dataset is loaded again so this notebook can run independently. Only protein-expression measurements are used as predictors. The experimental class is the target, while the biological mouse identifier is retained exclusively for group-aware evaluation.

In [2]:
from ucimlrepo import fetch_ucirepo

mice = fetch_ucirepo(id=342)

X = mice.data.features.drop(
    columns=["Genotype", "Treatment", "Behavior"]
)

y = mice.data.targets["class"]

groups = (
    mice.data.ids["MouseID"]
    .str.split("_")
    .str[0]
)

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)
print("Biological mice:", groups.nunique())

Predictor shape: (1080, 77)
Target shape: (1080,)
Biological mice: 72


## 2. Create a group-aware evaluation strategy

Three-fold stratified group cross-validation is used.

- **Group-aware:** all 15 measurements from one mouse remain together.
- **Stratified:** each fold should contain a similar distribution of the eight classes.
- **Cross-validation:** the model will eventually be evaluated three times using different test mice.
- **The number of folds depends mainly on:**
 1. Number of independent biological subjects.
 2. Number of subjects in the smallest class.
 3. Having enough mice for both training and reliable testing.

In [6]:
from sklearn.model_selection import StratifiedGroupKFold

group_splitter = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

first_split = next(
    group_splitter.split(X, y, groups=groups)
)

train_indices, test_indices = first_split

train_mice = set(groups.iloc[train_indices])
test_mice = set(groups.iloc[test_indices])

print("Training measurements:", len(train_indices))
print("Testing measurements:", len(test_indices))

print("Training mice:", len(train_mice))
print("Testing mice:", len(test_mice))

print("Mice present in both sets:", train_mice.intersection(test_mice))

Training measurements: 720
Testing measurements: 360
Training mice: 48
Testing mice: 24
Mice present in both sets: set()


## 3. Define the preprocessing steps

Missing protein measurements will be replaced with the median value of the corresponding protein. For every cross-validation fold, the median must be calculated using the training mice only and then applied to both the training and testing measurements.

In [7]:
from sklearn.impute import SimpleImputer

median_imputer = SimpleImputer(
    strategy="median"
)

### Standardize protein measurements

The protein variables have different numerical ranges. Standardization places them on comparable scales so that proteins with larger numerical values do not influence the model simply because of their measurement scale.

For every fold, the mean and standard deviation will be learned from the training mice only and then applied to both training and testing measurements.

In [8]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()